#  HT_04 — SAW Implementation

Benefit/cost normalization and weighted score computation.

*Related: [Methodology](../docs/methodology.md)*


In [ ]:
# Repo root on sys.path so `src` imports work
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.config import BENEFIT, CRITERIA, COST, WEIGHTS
from src.data.cleaner import clean
from src.data.loader import load_raw
from src.data.transformer import transform
from src.dss.saw import calculate_score, normalize


## 1. Criteria Configuration


In [ ]:
weights_df = pd.DataFrame({'weight': WEIGHTS, 'type': ['Benefit' if c in BENEFIT else 'Cost' for c in WEIGHTS]})
weights_df.loc['TOTAL', 'weight'] = weights_df['weight'].sum()
weights_df

## 2. Prepare Data


In [ ]:
df = transform(clean(load_raw('../data/raw/cleve.mod')))
print(f'{len(df)} patients ready')

## 3. Normalize Decision Matrix

- **Benefit:** `r = x / max(x)`
- **Cost:** `r = min(x) / x`


In [ ]:
norm = normalize(df, CRITERIA, BENEFIT)
norm.head()

In [ ]:
assert norm.min().min() >= 0 and norm.max().max() <= 1
print('All normalized values within [0, 1].')

## 4. Weighted Score


In [ ]:
df['score'] = calculate_score(norm, WEIGHTS).round(4)
df['score'].describe().to_frame('score').T

## 5. Manual Walkthrough (first patient)


In [ ]:
patient = norm.iloc[0]
walk = pd.DataFrame({'normalized': patient, 'weight': pd.Series(WEIGHTS)})
walk['contribution'] = walk['normalized'] * walk['weight']
walk.loc['TOTAL', 'contribution'] = walk['contribution'].sum()
walk

## Summary

- Weights sum to exactly 1.00
- Normalization bounds all values to [0, 1]
- Score = weighted sum of normalized criteria
